
## Data ingestion to Bronze layer

### Importing dependencies

In [0]:
import pyspark.sql.types as st

### Creating dataframe

In [0]:
hi_small_trans_schema = st.StructType([
    st.StructField("Timestamp", st.StringType(), True),
    st.StructField("From Bank", st.IntegerType(), True),
    st.StructField("Account2", st.StringType(), True),
    st.StructField("To Bank", st.IntegerType(), True),
    st.StructField("Account4", st.StringType(), True),
    st.StructField("Amount Received", st.DoubleType(), True),
    st.StructField("Receiving Currency", st.StringType(), True),
    st.StructField("Amount Paid", st.DoubleType(), True),
    st.StructField("Payment Currency", st.StringType(), True),
    st.StructField("Payment Format", st.StringType(), True),
    st.StructField("Is Laundering", st.IntegerType(), True)
])

hi_small_trans_file_path = "/Volumes/aml_transactions/aml_transactions_schema/source_files/HI-Small_Trans.csv"

column_mapping = {
    "Timestamp" : "timestamp",
    "From Bank" : "from_bank",
    "Account2" : "from_account",
    "To Bank" : "to_bank",
    "Account4" : "to_account",
    "Amount Received" : "amount_received",
    "Receiving Currency" : "receiving_currency",
    "Amount Paid" : "amount_paid",
    "Payment Currency" : "payment_currency",
    "Payment Format" : "payment_format",
    "Is Laundering" : "is_laundering"
}

df = (
    spark.read.format("csv")
    .option("header", "true")
    .schema(hi_small_trans_schema)
    .load(hi_small_trans_file_path)
).withColumnsRenamed(column_mapping)

timestamp,from_bank,from_account,to_bank,to_account,amount_received,receiving_currency,amount_paid,payment_currency,payment_format,is_laundering
2022/09/01 00:20,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0
2022/09/01 00:20,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0
2022/09/01 00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0
2022/09/01 00:02,12,8000F5030,12,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0
2022/09/01 00:06,10,8000F5200,10,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0


### Ingesting data to Bronze layer

In [0]:
bronze_aml_transactions_path = "aml_transactions.aml_transactions_schema.bronze_aml_transactions"

df.write.format("delta").mode("overwrite").saveAsTable(bronze_aml_transactions_path)

### Post ingestion validations

In [0]:
%sql
Select
    count(*) as row_count
from aml_transactions.aml_transactions_schema.bronze_aml_transactions

row_count
5078345


In [0]:
%sql
describe table aml_transactions.aml_transactions_schema.bronze_aml_transactions

col_name,data_type,comment
timestamp,string,null
from_bank,int,null
from_account,string,null
to_bank,int,null
to_account,string,null
amount_received,double,null
receiving_currency,string,null
amount_paid,double,null
payment_currency,string,null
payment_format,string,null
